In [ ]:
# ============================================================================
# 01_synthea_generate  (Airflow Option-B port)
# ----------------------------------------------------------------------------
# Parameter contract injected by the DAG's `generate` mapped task
# (build_generate_params): run_id_root + the cohort fields + stage. The
# remaining Synthea knobs (modules/seed/export flags) are resolved from
# dataset_id via COHORT_CONFIG below, so the orchestration layer only has to
# pass the cohort identity.
# ============================================================================

# PARAMETERS CELL ************
run_id_root   = ""              # run grouping key (utcnow yyyyMMddHHmmss) from set_run_id
dataset_id    = "ma_diabetes"   # logical cohort name (= cohort_id)
patient_count = 1000            # -p
state         = "Massachusetts" # final positional arg to synthea
stage         = "generate"      # run_state stage key


In [ ]:
# ---------------------------------------------------------------------------
# run_state skip/restart helpers (gold.control.run_state)
# Paste-in contract from include/run_state_helpers.py. The DAG never queries the
# lakehouse; each notebook self-records RUNNING -> SUCCEEDED/FAILED and self-skips
# a (run_id_root, cohort_id, stage) triple that already SUCCEEDED.
# ---------------------------------------------------------------------------
from datetime import datetime, timezone

RUN_STATE_TABLE = "lh_synthea_gold.control.run_state"


def already_succeeded(run_id_root, cohort_id, stage):
    """True if this (run_id_root, cohort_id, stage) already completed."""
    if not run_id_root or not spark.catalog.tableExists(RUN_STATE_TABLE):
        return False
    df = spark.sql(
        f"""
        SELECT 1 FROM {RUN_STATE_TABLE}
        WHERE run_id_root = '{run_id_root}'
          AND cohort_id   = '{cohort_id}'
          AND stage       = '{stage}'
          AND status      = 'SUCCEEDED'
        LIMIT 1
        """
    )
    return df.count() > 0


def mark(run_id_root, cohort_id, stage, status, error=None):
    """Idempotent UPSERT of a run_state row for this triple via MERGE."""
    if not run_id_root or not spark.catalog.tableExists(RUN_STATE_TABLE):
        return
    now = datetime.now(timezone.utc)
    err = (error or "").replace("'", "''")[:4000]
    spark.sql(
        f"""
        MERGE INTO {RUN_STATE_TABLE} AS t
        USING (
            SELECT
                '{run_id_root}' AS run_id_root,
                '{cohort_id}'   AS cohort_id,
                '{stage}'       AS stage,
                '{status}'      AS status,
                TIMESTAMP('{now.isoformat()}') AS ts,
                '{err}'         AS error
        ) AS s
        ON  t.run_id_root = s.run_id_root
        AND t.cohort_id   = s.cohort_id
        AND t.stage       = s.stage
        WHEN MATCHED THEN UPDATE SET
            t.status   = s.status,
            t.attempt  = COALESCE(t.attempt, 0) + CASE WHEN s.status = 'RUNNING' THEN 1 ELSE 0 END,
            t.started_ts = CASE WHEN s.status = 'RUNNING' THEN s.ts ELSE t.started_ts END,
            t.ended_ts   = CASE WHEN s.status IN ('SUCCEEDED','FAILED') THEN s.ts ELSE t.ended_ts END,
            t.error      = CASE WHEN s.status = 'FAILED' THEN s.error ELSE NULL END
        WHEN NOT MATCHED THEN INSERT (
            run_id_root, cohort_id, stage, status, attempt, started_ts, ended_ts, error
        ) VALUES (
            s.run_id_root, s.cohort_id, s.stage, s.status, 1, s.ts, NULL,
            CASE WHEN s.status = 'FAILED' THEN s.error ELSE NULL END
        )
        """
    )

import json, uuid, datetime as dt, os

# --- resolve the full Synthea config for this cohort (faithful to the source
#     pipeline's `datasets` array) -------------------------------------------
COHORT_CONFIG = {
    "ma_diabetes":        {"modules": "diabetes",                                   "seed": 100001, "fhir_export": True,  "csv_export": True},
    "oncology":           {"modules": "breast_cancer,colorectal_cancer,lung_cancer","seed": 100002, "fhir_export": True,  "csv_export": True},
    "claims_cpcds":       {"modules": "",                                           "seed": 100003, "fhir_export": False, "csv_export": True},
    "ehr_fhir":           {"modules": "",                                           "seed": 100004, "fhir_export": True,  "csv_export": False},
    "sdoh":               {"modules": "homelessness,food_insecurity,unemployment",  "seed": 100005, "fhir_export": True,  "csv_export": True},
    "houston_geo":        {"modules": "",                                           "seed": 100006, "fhir_export": True,  "csv_export": True},
    "provider_directory": {"modules": "",                                           "seed": 100007, "fhir_export": False, "csv_export": True},
    "covid_national":     {"modules": "covid19",                                    "seed": 100008, "fhir_export": True,  "csv_export": True},
}
_cfg = COHORT_CONFIG.get(dataset_id, {})
modules              = _cfg.get("modules", "")
seed                 = _cfg.get("seed", 12345)
fhir_export          = _cfg.get("fhir_export", True)
csv_export           = _cfg.get("csv_export", True)
clinical_note_export = _cfg.get("clinical_note_export", False)

# --- preserved run_id contract: run_id = run_id_root + "-" + dataset_id -----
cohort_id = dataset_id
run_id    = f"{run_id_root}-{dataset_id}" if run_id_root else f"interactive-{dataset_id}"
# The source body computes `run_id = run_id_override or uuid4()`; feed it our
# deterministic run_id so the generated output path matches what silver reads.
run_id_override = run_id

# --- per-cohort skip on re-run ---------------------------------------------
if already_succeeded(run_id_root, cohort_id, stage):
    mssparkutils.notebook.exit(json.dumps({"status": "SKIPPED", "run_id": run_id}))
mark(run_id_root, cohort_id, stage, "RUNNING")


In [ ]:
# The original stage body is wrapped so that a thrown exception is recorded
# as FAILED in run_state, while a normal/early return is recorded SUCCEEDED.
def _run_body():
    # CODE CELL ******************
    import json, uuid, datetime as dt, os
    from pyspark.sql import SparkSession

    spark = SparkSession.builder.getOrCreate()

    run_id     = run_id_override or str(uuid.uuid4())
    started_at = dt.datetime.utcnow().isoformat() + "Z"

    # Resolve all paths relative to the attached lakehouse (lh_synthea_bronze)
    local_root = "/lakehouse/default/Files"
    out_dir    = f"{local_root}/raw/{dataset_id}/{run_id}"
    jar_dir    = f"{local_root}/_bin"
    jar_path   = f"{jar_dir}/synthea-with-dependencies.jar"

    print(f"[run] dataset_id={dataset_id} run_id={run_id} patients={patient_count} state={state} seed={seed}")
    print(f"[run] output -> {out_dir}")

    # ---- skip-if-already-generated guard (idempotent / resumable) -------------
    # If this dataset_id+run_id was already generated (manifest present), reuse it
    # and exit immediately instead of re-running Synthea. This makes the pipeline
    # resumable: re-running with the same run_id_root skips completed cohorts.
    _existing_manifest = f"{out_dir}/_manifest.json"
    if os.path.exists(_existing_manifest):
        print(f"[skip] manifest already present -> {_existing_manifest}; skipping generation")
        with open(_existing_manifest) as _mf:
            _existing = _mf.read()
        return (_existing)
    # ---------------------------------------------------------------------------

    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(jar_dir, exist_ok=True)

    # Build the modules flag (Synthea uses -m "module1,module2"; omit if empty)
    modules_flag = f'-m "{modules}"' if modules else ""

    # Build optional exporter flags (Synthea defaults to FHIR on, CSV off)
    fhir_flag = f"--exporter.fhir.export={'true' if fhir_export else 'false'}"
    csv_flag  = f"--exporter.csv.export={'true' if csv_export else 'false'}"
    note_flag = f"--exporter.clinical_note.export={'true' if clinical_note_export else 'false'}"

    # Use a single shell call so the jar download + java run share one bash session.
    # Note: %%sh lines must remain unindented and at the top of a cell. We emulate
    # it via subprocess so we can interpolate Python parameters cleanly.
    import subprocess, shlex, sys

    bash = f"""
    set -euo pipefail

    JAR="{jar_path}"
    OUT="{out_dir}"

    SYNTHEA_VERSION="v3.2.0"
    if [ ! -f "$JAR" ]; then
      echo "[bash] Downloading Synthea jar ($SYNTHEA_VERSION, last Java 11-compatible release)..."
      mkdir -p "$(dirname "$JAR")"
      curl -fsSL -o "$JAR" \
        "https://github.com/synthetichealth/synthea/releases/download/$SYNTHEA_VERSION/synthea-with-dependencies.jar"
    fi

    mkdir -p "$OUT"

    java -Xmx6g -jar "$JAR" \
      -p {patient_count} \
      -s {seed} \
      {fhir_flag} \
      {csv_flag} \
      {note_flag} \
      --exporter.baseDirectory="$OUT" \
      --exporter.csv.folder_per_run=false \
      --exporter.fhir.bulk_data=true \
      {modules_flag} \
      "{state}"
    """

    print("[run] launching synthea...")
    proc = subprocess.run(["bash", "-c", bash], capture_output=True, text=True)
    sys.stdout.write(proc.stdout[-4000:])
    sys.stderr.write(proc.stderr[-4000:])
    if proc.returncode != 0:
        raise RuntimeError(f"Synthea generation failed (exit={proc.returncode})")

    # Synthea writes:
    #   $OUT/csv/*.csv
    #   $OUT/fhir/*.ndjson
    csv_dir  = f"{out_dir}/csv"
    fhir_dir = f"{out_dir}/fhir"

    def list_files(d):
        if not os.path.isdir(d):
            return []
        return sorted(os.listdir(d))

    csv_files  = list_files(csv_dir)
    fhir_files = list_files(fhir_dir)

    finished_at = dt.datetime.utcnow().isoformat() + "Z"

    manifest = {
        "dataset_id":    dataset_id,
        "run_id":        run_id,
        "patient_count": patient_count,
        "state":         state,
        "modules":       modules,
        "seed":          seed,
        "started_at":    started_at,
        "finished_at":   finished_at,
        "output_paths": {
            "root": f"Files/raw/{dataset_id}/{run_id}",
            "csv":  f"Files/raw/{dataset_id}/{run_id}/csv",
            "fhir": f"Files/raw/{dataset_id}/{run_id}/fhir",
        },
        "csv_files":  csv_files,
        "fhir_files": fhir_files,
    }

    manifest_path = f"{out_dir}/_manifest.json"
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    print(f"[run] manifest written -> {manifest_path}")
    print(json.dumps(manifest, indent=2))

    return (json.dumps(manifest))



try:
    _result = _run_body()
except Exception as _e:
    mark(run_id_root, cohort_id, stage, "FAILED", error=str(_e))
    raise
mark(run_id_root, cohort_id, stage, "SUCCEEDED")
mssparkutils.notebook.exit(_result)
